# ✅ Student Marks Prediction — With Pipeline

Before pipelines, our workflow required:
- **Remembering** every preprocessing step
- **Matching columns** manually during inference
- **Scaling** separately, with a separate saved file
- **Saving multiple objects** just to make one prediction

Let's fix that properly. 👇

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 2. Load Dataset

In [ ]:
df = pd.read_csv("data.csv")
X = df.drop(columns=["final_score"])
y = df["final_score"]
print("Shape:", df.shape)

## 3. Define Column Groups

In [ ]:
numerical_cols   = ["study_hours", "attendance_percentage", "sleep_hours", "previous_exam_score"]
categorical_cols = ["internet_access", "parental_education", "extracurricular_activity", "part_time_job", "motivation_level"]

## 4. Build Sub-Pipelines

Each column type gets its own mini-pipeline.

In [ ]:
# Numerical: impute → scale
numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler",  StandardScaler())
])

# Categorical: impute → one-hot encode
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

> **Note on scaling + Random Forest:** Random Forest doesn't *mathematically* need scaling, but we include `StandardScaler` here intentionally.  
> In real pipelines you'll often swap in models that *do* require it (e.g. Linear Regression, SVM, KNN). Keeping scaling in the pipeline makes the workflow model-agnostic and production-safe.

## 5. Combine with ColumnTransformer

`ColumnTransformer` applies each pipeline to the right columns automatically.

In [ ]:
preprocessor = ColumnTransformer([
    ("num", numerical_pipeline,   numerical_cols),
    ("cat", categorical_pipeline, categorical_cols)
])

## 6. Build the Final Pipeline

> **Preprocessing + Model = one single object.**  
> This is the core idea of sklearn Pipelines.

In [ ]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model",        RandomForestRegressor(n_estimators=100, random_state=42))
])

## 7. Train

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# One call trains everything: imputer, scaler, encoder, and model
pipeline.fit(X_train, y_train)
print("Pipeline trained!")

## 8. Evaluate

In [ ]:
y_pred = pipeline.predict(X_test)

print(f"MAE  : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, y_pred)):.2f}")
print(f"R²   : {r2_score(y_test, y_pred):.4f}")

## 9. Generalization Check

Are we overfitting? Compare Train R² vs Test R².

In [ ]:
train_score = pipeline.score(X_train, y_train)
test_score  = pipeline.score(X_test,  y_test)

print(f"Train R²: {train_score:.4f}")
print(f"Test  R²: {test_score:.4f}")

# Similar values → good generalization
# Large gap       → overfitting

## 10. Save the Complete Pipeline

One file. Contains everything: imputer + scaler + encoder + model.

In [ ]:
joblib.dump(pipeline, "with_pipeline.pkl")
print("Complete pipeline saved!")

---

## 📊 Summary — Pipeline vs Manual Workflow

| Task | Without Pipeline | With Pipeline |
|---|---|---|
| Missing value handling | Manual | ✅ Automatic |
| Encoding | Manual | ✅ Automatic |
| Scaling | Manual | ✅ Automatic |
| Prediction workflow | Complex | ✅ Simple |
| Deployment readiness | Difficult | ✅ Easier |

> Next: `test_with_pipeline.ipynb` — inference in just 3 lines. 🚀